In [21]:
import pandas as pd
data = "/data/projects/punim0478/setiawand/bible-nmt/results/gemini-2.5-flash-engBBE-dhao/parallel_full/gemini-2.5-flash_aligned-eng-engBBE-ot_eng_nfa.csv"
input_data = "/data/projects/punim0478/setiawand/bible-nmt/ebible-corpus/dhao-eng/engBBE/aligned-eng-engBBE-ot.csv"

# Load both dataframes
output = pd.read_csv(data)
input_df = pd.read_csv(input_data)

# Take only first 500 rows of input dataframe
input_df = input_df.head(500)

# Check for duplicate rows in the output dataframe
print(f"Total rows before removing duplicates: {len(output)}")

# Check which columns have duplicates
duplicate_counts = {}
for col in output.columns:
    duplicates = output[output.duplicated(subset=[col], keep=False)]
    if len(duplicates) > 0:
        duplicate_counts[col] = len(duplicates)

print("Duplicate counts by column:")
for col, count in duplicate_counts.items():
    print(f"  {col}: {count} rows")

# Remove duplicate rows (keeping the first occurrence)
output_no_duplicates = output.drop_duplicates()
print(f"Total rows after removing duplicates: {len(output_no_duplicates)}")
print(f"Removed {len(output) - len(output_no_duplicates)} duplicate rows")

# Replace the original dataframe with the deduplicated one
output = output_no_duplicates

# Keep only rows in output that exist in input_df based on source_text
output = output[output['src_text'].isin(input_df['source_text'])]
print(f"Total rows after removing rows not in input: {len(output)}")

# For duplicate source texts, keep only the one with the highest chrfpp_improvement
print(f"Rows before handling src_text duplicates: {len(output)}")
duplicate_src_count = output.duplicated(subset=['src_text']).sum()
print(f"Number of duplicate src_text rows: {duplicate_src_count}")

# Sort by chrfpp_improvement in descending order, then drop duplicates keeping the first (highest score)
output = output.sort_values('chrfpp_improvement', ascending=False).drop_duplicates(subset=['src_text'], keep='first')
print(f"Rows after keeping highest chrfpp_improvement for each src_text: {len(output)}")

# Create a mapping from source_text to the original order in input_df
input_order = {text: i for i, text in enumerate(input_df['source_text'])}

# Add a sorting column based on the original order
output['sort_order'] = output['src_text'].map(input_order)

# Sort the output dataframe to match the input order
output = output.sort_values('sort_order').drop('sort_order', axis=1)

print(f"Final output dataframe has {len(output)} rows")
print(f"Total rows after removing rows not in input: {len(output)}")

# Create a mapping from source_text to the original order in input_df
input_order = {text: i for i, text in enumerate(input_df['source_text'])}

# Add a sorting column based on the original order
output['sort_order'] = output['src_text'].map(input_order)

# Sort the output dataframe to match the input order
output = output.sort_values('sort_order').drop('sort_order', axis=1)

print(f"Final output dataframe has {len(output)} rows")


Total rows before removing duplicates: 579
Duplicate counts by column:
  src_text: 123 rows
  tgt_text: 123 rows
  pred_tgt_text: 123 rows
  post_edited_tgt_txt: 119 rows
  src_lang: 579 rows
  tgt_lang: 579 rows
  src_lang_name: 579 rows
  tgt_lang_name: 579 rows
  spbleu_improvement: 124 rows
  chrf3_improvement: 124 rows
  chrfpp_improvement: 125 rows
Total rows after removing duplicates: 505
Removed 74 duplicate rows
Total rows after removing rows not in input: 505
Rows before handling src_text duplicates: 505
Number of duplicate src_text rows: 5
Rows after keeping highest chrfpp_improvement for each src_text: 500
Final output dataframe has 500 rows
Total rows after removing rows not in input: 500
Final output dataframe has 500 rows


In [27]:
df = pd.read_csv("/data/projects/punim0478/setiawand/bible-nmt/results/gemini-2.5-flash-engBBE-dhao/parallel_full+nt/gemini-2.5-flash_aligned-eng-engBBE-ot_eng_nfa.csv")
# Display the dataframe
print(f"Average chrfpp_improvement: {df['chrfpp_improvement'].mean():.4f}")


Average chrfpp_improvement: 6.6562


In [2]:
import pandas as pd
import os

# Load the data
file_path = "/data/projects/punim0478/setiawand/bible-nmt/ebible-corpus/dhao-eng/engBBE/aligned-eng-engBBE-all.csv"
df = pd.read_csv(file_path)

# Calculate the length difference between source and target text
df['source_len'] = df['source_text'].str.len()
df['target_len'] = df['target_text'].str.len()
df['len_diff'] = abs(df['target_len'] - df['source_len'])

# Sort by the length difference in descending order and get top 5
top_diff_rows = df.sort_values(by='len_diff', ascending=False).head(5)

# Display the results
top_diff_rows[['verse', 'source_text', 'target_text', 'source_len', 'target_len', 'len_diff']]


,verse,source_text,target_text,source_len,target_len,len_diff
8066,HEB 7:1-3,"For this Melchizedek, the king of Salem, a pri...","Uru èèna, Melkisedek jꞌajꞌi dhèu aae kota Sale...",474,1222,748
6454,1CO 1:21-22,"For because, by the purpose of God, the world,...",Mamèu Ama Lamatua lènge aaꞌi mèu-mèu. Masi ka ...,292,984,692
5628,ACT 16:37,"But Paul said to them, They have given us who ...","Tadèngi rare sèmi èèna, ka Paulus noꞌo boe sèm...",210,751,541
8357,1PE 1:1-2,"Peter, an Apostle of Jesus Christ, to the sain...","Sasuri ne, nèti jaꞌa, Petrus, dhèu pajuu-paleh...",274,796,522
8919,REV 15:3,"And they give the song of Moses, the servant o...",Ka ra sodꞌa lii sodꞌa èci dhu Musa suri. Musa ...,213,714,501


In [15]:
### Count how many average text length in the dataset
import pandas as pd
import string
from nltk.tokenize import word_tokenize
df = pd.read_csv("/data/projects/punim0478/setiawand/bible-nmt/ebible-corpus/dhao-eng/engwebp/aligned-eng-engwebp-nt.csv")
df_2 = pd.read_csv("/data/projects/punim0478/setiawand/bible-nmt/lexical-resource/eng-dhao/dhao-english-parallel-full.csv")
df = pd.concat([df, df_2])
df
avg_sen_length = []
for _, datum in df.iterrows():
    filtered_words = [i for i in word_tokenize(str(datum["source_text"])) if i not in string.punctuation]
    avg_sen_length.append(len(filtered_words))
print(sum(avg_sen_length) / len(avg_sen_length))


21.214888452202057
